# 🚀 Tái Lập SMC & RS-SMC (Bảng 8) Trên Google Colab (GPU A100 / L4 / T4)

Notebook này được tối ưu hóa toàn diện cho **Google Colab (A100 / L4 / T4)** theo đúng chuẩn kiến trúc của `LiDAR_Table2_Replication_Colab.ipynb`:
- **Chuyên biệt hóa tái lập & mở rộng Bảng 8 (Table 8)**: Khảo sát tương tác đa hạt mục tiêu của **Sequential Monte Carlo (SMC)**, **Randomized Smoothing SMC (RS-SMC)**, và so sánh đối chiếu với **Best-of-N (BoN)** & **LiDAR**.
- **Cấu trúc lưu trữ tức thì (Real-time Drive Persistence)**: Toàn bộ ảnh mẫu, điểm số và thông số được lưu trực tiếp ra Google Drive thông qua **Symbolic Link** (`Target_samples/` -> `Google Drive/RS-LiDAR/Target_samples/`). Không lo mất dữ liệu khi mất kết nối mạng.
- **Tiếp tục thông minh (`--resume`) & Kế thừa tiến độ**: Tự động nhận diện các prompt đã hoàn thành để bỏ qua (Skip 0.00s) và chạy tiếp prompt còn thiếu.
- **Đặt tên thư mục cực kỳ hệ thống**: Mọi siêu tham số quan trọng (`MODEL`, `METHOD`, `N`, `SIGMA`, `M`, `CRN`, `FREQ`, `LAMBDA`, `SOLVER`, `SCALE`, `SEED`, `PROMPTS`) đều được mã hóa chi tiết trong tên thư mục.

---

### 💡 Trực Giác Toán Học: Tại Sao Làm Mịn Reward Trên Ước Lượng Tweedie $\hat{x}_0$?
1. **Độ lệch Tweedie (Tweedie Approximation Drift)**: Ở các bước khử nhiễu trung gian $t \in [20, 80]$, ước lượng Tweedie $\hat{x}_0(x_t) = \mathbb{E}[x_0 \mid x_t]$ là một xấp xỉ mờ (blurry expectation), thiếu chi tiết tần số cao và sai lệch phân phối so với ảnh hội tụ cuối $x_0^*$.
2. **Hàm Reward Thô $R_{\text{raw}}$ Rất Nhạy Cảm**: Các mô hình như ImageReward có độ dốc Lipschitz $L_{\text{raw}}$ lớn và nhiều gai nhọn đối kháng. Sai lệch nhỏ trong ảnh Tweedie khiến điểm reward dao động gắt gao:
   - *Premature Pruning*: Loại bỏ nhầm hạt tiềm năng chỉ vì ảnh mờ Tweedie tạm rơi vào vùng trũng điểm số.
   - *Spurious Collapse*: Nhân bản hạt lỗi ra toàn bộ quần thể ($ESS \to 1.0$) vì một đốm nhiễu vô tình kích hoạt điểm cao giả tạo.
3. **Randomized Smoothing Giải Quyết Triệt Để**: Tích chập Gauss $\hat{R}_{\sigma}(x) = \mathbb{E}_{\epsilon}[R(x+\epsilon)]$ làm phẳng toàn bộ các gai nhọn, giới hạn độ dốc Lipschitz toàn cục $\|\nabla \hat{R}_{\sigma}\| \le \frac{\sqrt{2/\pi}}{\sigma} \|R\|_{\infty}$. Nhờ đó:
   $$|\hat{R}_{\sigma}(\hat{x}_0) - \hat{R}_{\sigma}(x_0^*)| \le L_{\sigma} \|\hat{x}_0 - x_0^*\|$$
   Reward của ước lượng Tweedie không còn bị giật cục, bám sát điểm số thực tế của quỹ đạo đích, giúp SMC tái chọn mẫu chính xác vào các vùng nghiệm chất lượng cao.

4. **Lập Lịch Co Giảm Giải Tích (Tweedie Analytical Annealing Schedule)**:
   - Theo công thức Tweedie, phương sai hậu nghiệm của $\hat{x}_0$ tỷ lệ với $\psi^2(t) = \frac{1-\bar{\alpha}_t}{\bar{\alpha}_t}$.
   - Khi khử nhiễu từ bước $t_{\text{start}} = 20$ về $t_{\text{end}} = 80$, độ bất định Tweedie $\psi(t)$ suy giảm giải tích từ $2.032$ xuống $0.222$ (co lại $89.1\%$).
   - RS-SMC lập lịch co giảm mượt mà:
     $$\sigma(t) = \sigma_{\text{init}} \cdot \frac{\psi(t) - \psi(t_{\text{end}})}{\psi(t_{\text{start}}) - \psi(t_{\text{end}})}$$
   - Khởi đầu tối ưu: $\sigma_{\text{init}} = 0.25$ (SD 1.5) hoặc $0.15$ (SDXL).
   - Tại các bước $t \le t_{\text{end}}$ và terminal step $i=99$, $\sigma(t) \equiv 0.0$ triệt tiêu hoàn toàn, bảo toàn 100% độ sắc nét ảnh và chấm điểm ImageReward/HPSv2 chuẩn xác khi tuyển chọn Best-of-N.


## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> Chọn GPU (A100, L4 hoặc T4)**.


In [ ]:
import torch, sys, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    elif vram_gb >= 12.0:
        print("✅ GPU phù hợp cho thực nghiệm SD 1.5 và SDXL (chạy chế độ chunking).")
    else:
        print("ℹ️ GPU cơ bản (T4). Khuyến nghị chạy SD 1.5 với dải hạt N <= 8.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi


## 2. Gắn Kết Google Drive & Thiết Lập Lưu Trữ Tức Thì (Real-time Persistence)
Toàn bộ kết quả thực nghiệm sẽ được ghi trực tiếp vào `My Drive/RS-LiDAR/Target_samples/` thông qua Symbolic Link. Sau mỗi prompt hoàn thành, file `.png` và `results.json` được lưu vĩnh viễn trên Drive.


In [ ]:
import os, shutil, subprocess
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)

# 2. Clone hoặc kéo cập nhật repo RS-LiDAR tự động
REPO_DIR = "/content/RS-LiDAR"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/leekwanreal/RS-LiDAR.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/LiDAR_sampling.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}

# 3. Tạo Symbolic Links an toàn (Tuyệt đối không xóa dữ liệu cũ, tự động migrate nếu có file cục bộ)
for folder_name in ["Lookahead_samples", "Target_samples"]:
    link_path = os.path.join(WORKDIR, folder_name)
    drive_target = os.path.join(DRIVE_DIR, folder_name)
    os.makedirs(drive_target, exist_ok=True)

    if os.path.islink(link_path):
        if not os.path.exists(link_path):
            os.remove(link_path)
            os.symlink(drive_target, link_path)
    elif os.path.exists(link_path):
        print(f"📦 Phát hiện thư mục cục bộ {folder_name}, đang đồng bộ an toàn sang Google Drive...")
        for item in os.listdir(link_path):
            src_item = os.path.join(link_path, item)
            dst_item = os.path.join(drive_target, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        shutil.rmtree(link_path)
        os.symlink(drive_target, link_path)
    else:
        os.symlink(drive_target, link_path)

print("✅ Hệ thống Symlink đã sẵn sàng 100%! Toàn bộ dữ liệu được bảo lưu vĩnh viễn trên Drive:")
print("  📁", DRIVE_DIR)


## 3. Cài Đặt Môi Trường Chuẩn Xác (100% Trơn Tru & Không Xung Đột)
Cài đặt bộ thư viện đồng bộ, tắt backend TensorFlow để chống xung đột Protobuf, và tải vocab HPSv2.


In [ ]:
import os, urllib.request

# 1. Tắt backend TensorFlow để loại bỏ xung đột Protobuf runtime_version
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Nâng cấp protobuf và cài đặt các thư viện tương thích PyTorch 2.x
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Đảm bảo file từ điển OpenCLIP cho HPSv2
import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    print("📥 Đang tải file BPE vocab cho HPSv2...")
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("✅ Cài đặt môi trường hoàn tất 100%!")


## 4. Cấu Hình Siêu Tham Số Hệ Thống Cho SMC & RS-SMC (Bảng 8)
Chọn mô hình, phương pháp và siêu tham số. Thư mục đầu ra được tự động đặt tên **cực kỳ hệ thống**, mã hóa toàn bộ cấu hình thực nghiệm.


In [ ]:
# @title ⚙️ Cấu Hình Thực Nghiệm & Kiểm Tra Tiến Độ Trên Drive
import glob, os, json

# ==============================================================================
# 🎯 0. CHỌN CHẾ ĐỘ THỰC THI (EXECUTION MODE)
# ==============================================================================
EXECUTION_MODE = "SINGLE_RUN" # @param ["SINGLE_RUN", "SWEEP_TABLE_8"]

# ==============================================================================
# 🎯 1. CHỌN MÔ HÌNH VÀ PHƯƠNG PHÁP LẤY MẪU
# ==============================================================================
MODEL_CHOICE = "Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)" # @param ["Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)", "SDXL Base 1.0 (stabilityai/stable-diffusion-xl-base-1.0)"]
METHOD = "RS-SMC + LiDAR" # @param ["RS-SMC", "Vanilla-SMC", "RS-SMC + LiDAR", "RS-SMC + RS-LiDAR", "Vanilla-SMC + LiDAR", "Vanilla-SMC + RS-LiDAR", "LiDAR", "BoN"]
NUM_PARTICLES = 4 # @param [1, 2, 4, 8] {type:"raw"}

# ==============================================================================
# 🧪 2. CẤU HÌNH RANDOMIZED SMOOTHING & ANNEALING (CHO RS-SMC)
# ==============================================================================
SMC_SIGMA = 0.25 # @param {type:"number"}
# Gợi ý chuẩn thực nghiệm tối ưu (Tweedie Posterior Calibration): 0.20 - 0.30 cho SD 1.5; 0.10 - 0.20 cho SDXL (Khắc phục hoàn toàn lỗi làm mờ quá mức)
SMC_ANNEAL_SIGMA = True # @param {type:"boolean"}
# Bật Annealed RS-SMC: Tự động điều chỉnh sigma theo giải tích độ bất định Tweedie psi(t) = sqrt((1-alpha_bar_t)/alpha_bar_t)
SMC_ANNEAL_SCHEDULE = "tweedie_analytical" # @param ["tweedie_analytical", "cosine", "linear", "sqrt_noise"]
SMC_SIGMA_MIN = 0.01 # @param {type:"number"}
SMC_NUM_MC_SAMPLES = 4 # @param {type:"integer"}
# M = 4 là điểm cân bằng lý tưởng giữa hiệu năng và thời gian thực thi
SMC_USE_CRN = False # @param {type:"boolean"}
# Common Random Numbers: Khuyến nghị False (No-CRN) để dùng nhiễu độc lập, chống suy sụp đàn hạt (Particle Collapse) và tối ưu ImageReward/HPSv2
SMC_SMOOTHING_MODE = "vectorized" # @param ["vectorized", "loop"]
# "vectorized": Tính toán song song toàn bộ M*N ảnh trên GPU (Khuyến nghị trên A100/L4 để tăng tốc gấp 3-4 lần, tiết kiệm tối đa compute units)
# "loop": Vòng lặp tuần tự từng mẫu M (tiết kiệm VRAM tối đa cho GPU yếu T4)

# ==============================================================================
# 🔮 3. CẤU HÌNH TÁI SỬ DỤNG LOOKAHEAD PHASE 1 (CHO LIDAR & RS-LIDAR)
# ==============================================================================
# SMC chỉ chạy ở Phase 2 (Target Sampling). Khi kết hợp LiDAR/RS-LiDAR (--use_rag),
# hệ thống sẽ tái sử dụng trực tiếp các tập mẫu Lookahead Phase 1 đã chạy sẵn trên Drive!
LOOKAHEAD_STEPS = 5 # @param [5] {type:"raw"}
LOOKAHEAD_PARTICLES = 50 # @param [50] {type:"raw"}
LOOKAHEAD_SIGMA = 1.0 # @param {type:"number"} # Tham số sigma của Phase 1 RS-LiDAR (chuẩn: 1.0)
LOOKAHEAD_M = 4 # @param {type:"integer"} # Số mẫu Monte Carlo Phase 1 RS-LiDAR (chuẩn: 4)
LOOKAHEAD_USE_CRN = False # @param {type:"boolean"}
LOOKAHEAD_SEED = 100 # @param {type:"integer"}

# ==============================================================================
# ⚙️ 4. CẤU HÌNH SMC & QUÁ TRÌNH KHỬ NHIỄU (TABLE 8 STANDARD)
# ==============================================================================
NUM_INFERENCE_STEPS = 100 # @param {type:"integer"}
ETA = 1.0 # @param {type:"number"}
# eta = 1.0: Tương đương toán học 100% với DDPM (Chuẩn Table 2 & Table 8)
# Chuẩn Bảng 8: DDPM 100 bước (eta = 1.0)
RESAMPLE_FREQUENCY = 20 # @param {type:"integer"}
# Resample mỗi 20 bước khử nhiễu
FK_LMBDA = 10.0 # @param {type:"number"}
# Nhiệt độ Feynman-Kac chuẩn Bảng 8
RESAMPLE_T_START = 20 # @param {type:"integer"}
RESAMPLE_T_END = 80 # @param {type:"integer"}
SMC_TERMINAL_RESAMPLE = True # Luôn mặc định sample bước terminal (99) theo chuẩn tác giả Horvitz
SEED = 100 # @param {type:"integer"}
OVERWRITE = False # @param {type:"boolean"}
SAVE_INDIVIDUAL_IMAGES = True # @param {type:"boolean"}
# Bat buoc bat True de luu file anh PNG (samples/ va best_of_n_samples/) phuc vu cham diem GenEval

# ==============================================================================
# 🎯 5. CẤU HÌNH TẬP PROMPTS (PROMPT SUBSET)
# ==============================================================================
PROMPT_MODE = "ALL" # @param ["100_REPRESENTATIVE", "ALL", "FIRST_100", "QUICK_12"]

if PROMPT_MODE == "100_REPRESENTATIVE":
    MAX_PROMPT = 100
    PROMPT_INDICES_ARG = "stratified_100"
    SUBSET_TAG = "_100p"
elif PROMPT_MODE == "FIRST_100":
    MAX_PROMPT = 100
    PROMPT_INDICES_ARG = "0-99"
    SUBSET_TAG = "_first100p"
elif PROMPT_MODE == "QUICK_12":
    MAX_PROMPT = 12
    PROMPT_INDICES_ARG = "80,98,95,93,10,25,30,45,50,65,70,85"
    SUBSET_TAG = "_12p"
else:
    MAX_PROMPT = 553
    PROMPT_INDICES_ARG = None
    SUBSET_TAG = ""

# ==============================================================================
# 🏷️ 6. TỰ ĐỘNG THIẾT LẬP THAM SỐ THEO KIẾN TRÚC MÔ HÌNH
# ==============================================================================
is_sdxl = "xl" in MODEL_CHOICE.lower()
MODEL_TAG = "SDXL" if is_sdxl else "SD15"
MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0" if is_sdxl else "runwayml/stable-diffusion-v1-5"
SCALE = 8.0 if is_sdxl else 12.5

# Auto-dispatch sigma nếu dùng SDXL và đang để mặc định của SD 1.5
if is_sdxl and "RS-SMC" in METHOD and SMC_SIGMA == 0.25:
    SMC_SIGMA = 0.15
    print("⚡ Auto-dispatch SMC_SIGMA = 0.15 cho SDXL (Khoảng hiệu chuẩn tối ưu [0.10, 0.20])")

SOLVER_TAG = f"DDPM{NUM_INFERENCE_STEPS}" if ETA == 1.0 else f"DDIM{NUM_INFERENCE_STEPS}"

# ==============================================================================
# 🔮 TỰ ĐỘNG XÁC ĐỊNH & KIỂM TRA ĐƯỜNG DẪN LOOKAHEAD PHASE 1 TRÊN DRIVE
# ==============================================================================
is_lidar = "LiDAR" in METHOD
is_rs_lidar = "RS-LiDAR" in METHOD

if is_rs_lidar:
    crn_tag_lk = "_crn" if LOOKAHEAD_USE_CRN else ""
    LOOKAHEAD_TAG = f"RSLookahead_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{LOOKAHEAD_PARTICLES}_sig{LOOKAHEAD_SIGMA}_M{LOOKAHEAD_M}{crn_tag_lk}_seed{LOOKAHEAD_SEED}"
elif is_lidar:
    LOOKAHEAD_TAG = f"Lookahead_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{LOOKAHEAD_PARTICLES}_seed{LOOKAHEAD_SEED}"
else:
    LOOKAHEAD_TAG = ""

found_lookahead_dir = None
lookahead_count = 0
if LOOKAHEAD_TAG:
    candidate_lk_paths = [
        f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}",
        f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}{SUBSET_TAG}",
        f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}_100p",
    ]
    if 'DRIVE_DIR' in locals() and DRIVE_DIR:
        candidate_lk_paths.extend([
            f"{DRIVE_DIR}/Lookahead_samples/{LOOKAHEAD_TAG}",
            f"{DRIVE_DIR}/Lookahead_samples/{LOOKAHEAD_TAG}{SUBSET_TAG}",
            f"{DRIVE_DIR}/Lookahead_samples/{LOOKAHEAD_TAG}_100p",
            f"/content/drive/MyDrive/RS-LiDAR/Lookahead_samples/{LOOKAHEAD_TAG}",
            f"/content/drive/MyDrive/RS-LiDAR/Lookahead_samples/{LOOKAHEAD_TAG}_100p",
            f"/content/drive/MyDrive/RS_LiDAR/Lookahead_samples/{LOOKAHEAD_TAG}",
            f"/content/drive/MyDrive/RS_LiDAR/Lookahead_samples/{LOOKAHEAD_TAG}_100p",
        ])
    
    for c_path in candidate_lk_paths:
        if os.path.exists(c_path):
            n_latents = len(glob.glob(f"{c_path}/[0-9]*/samples/latent.pt"))
            if n_latents > 0:
                found_lookahead_dir = c_path
                lookahead_count = n_latents
                break

LOOKAHEAD_PATH_ARG = os.path.basename(found_lookahead_dir) if found_lookahead_dir else LOOKAHEAD_TAG

# ==============================================================================
# 📁 7. ĐẶT TÊN THƯ MỤC CỰC KỲ HỆ THỐNG (SYSTEMATIC NAMING CONVENTION)
# ==============================================================================
def build_systematic_run_name(model_tag, method, num_particles, scale, steps=100,
                              resample_freq=20, fk_lmbda=10.0,
                              sigma=0.25, num_mc=4, use_crn=False, use_anneal=False,
                              terminal_resample=True,
                              lk_sigma=1.0, lk_m=4, lk_crn=False,
                              seed=42, subset_tag="", eta=1.0):
    solver = f"DDPM{steps}" if eta == 1.0 else f"DDIM{steps}"
    scale_tag = f"s{scale}"
    term_tag = "" if terminal_resample else "_noTerminal"
    
    if method in ["RS-SMC", "rs_smc", "Annealed-RS-SMC", "Static-RS-SMC"]:
        crn_tag = "_crn" if use_crn else ""
        anneal_tag = "_anneal" if (use_anneal or method == "Annealed-RS-SMC") else ""
        base = f"RSSMC_{model_tag}_N{num_particles}_sig{sigma}_M{num_mc}{crn_tag}{anneal_tag}{term_tag}_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method == "RS-SMC + LiDAR":
        crn_tag = "_crn" if use_crn else ""
        anneal_tag = "_anneal" if use_anneal else ""
        base = f"RSSMC_{model_tag}_N{num_particles}_sig{sigma}_M{num_mc}{crn_tag}{anneal_tag}{term_tag}_LiDAR_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method == "RS-SMC + RS-LiDAR":
        crn_tag = "_crn" if use_crn else ""
        anneal_tag = "_anneal" if use_anneal else ""
        lk_crn_tag = "_crn" if lk_crn else ""
        base = f"RSSMC_{model_tag}_N{num_particles}_sig{sigma}_M{num_mc}{crn_tag}{anneal_tag}{term_tag}_RSLiDAR_sig{lk_sigma}_M{lk_m}{lk_crn_tag}_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["Vanilla-SMC", "vanilla_smc", "SMC", "smc"]:
        base = f"SMC_{model_tag}_N{num_particles}{term_tag}_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method == "Vanilla-SMC + LiDAR":
        base = f"SMC_{model_tag}_N{num_particles}{term_tag}_LiDAR_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method == "Vanilla-SMC + RS-LiDAR":
        lk_crn_tag = "_crn" if lk_crn else ""
        base = f"SMC_{model_tag}_N{num_particles}{term_tag}_RSLiDAR_sig{lk_sigma}_M{lk_m}{lk_crn_tag}_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["BoN", "bon"]:
        base = f"BoN_{model_tag}_N{num_particles}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["LiDAR", "lidar"]:
        base = f"LiDAR_{model_tag}_N{num_particles}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["RS-LiDAR", "rs_lidar"]:
        lk_crn_tag = "_crn" if lk_crn else ""
        base = f"RSLiDAR_{model_tag}_N{num_particles}_sig{lk_sigma}_M{lk_m}{lk_crn_tag}_{solver}_{scale_tag}_seed{seed}"
    else:
        sanitized = method.replace(" ", "").replace("+", "_")
        base = f"{sanitized}_{model_tag}_N{num_particles}_{solver}_{scale_tag}_seed{seed}"
        
    return f"{base}{subset_tag}"

RUN_NAME = build_systematic_run_name(
    model_tag=MODEL_TAG,
    method=METHOD,
    num_particles=NUM_PARTICLES,
    scale=SCALE,
    steps=NUM_INFERENCE_STEPS,
    resample_freq=RESAMPLE_FREQUENCY,
    fk_lmbda=FK_LMBDA,
    sigma=SMC_SIGMA,
    num_mc=SMC_NUM_MC_SAMPLES,
    use_crn=SMC_USE_CRN,
    use_anneal=SMC_ANNEAL_SIGMA,
    terminal_resample=SMC_TERMINAL_RESAMPLE,
    lk_sigma=LOOKAHEAD_SIGMA,
    lk_m=LOOKAHEAD_M,
    lk_crn=LOOKAHEAD_USE_CRN,
    seed=SEED,
    subset_tag=SUBSET_TAG
)

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"

# Kiểm tra tiến độ thực nghiệm đã có sẵn trên Google Drive
existing_prompts = len(glob.glob(f"{target_dir}/[0-9]*/results.json")) if os.path.exists(target_dir) else 0

print("=" * 70)
print(f"📊 CẤU HÌNH THỰC NGHIỆM ĐÃ THIẾT LẬP:")
print(f"  • Mô hình:              {MODEL_TAG} ({MODEL_NAME})")
print(f"  • Phương pháp:          {METHOD}")
print(f"  • Số hạt mục tiêu N:    {NUM_PARTICLES}")
print(f"  • Guidance Scale:       {SCALE}")
if "RS-SMC" in METHOD:
    print(f"  • Base Smoothing Sigma: {SMC_SIGMA}")
    print(f"  • Annealed RS-SMC:      {SMC_ANNEAL_SIGMA} ({SMC_ANNEAL_SCHEDULE})")
    print(f"  • Monte Carlo M:        {SMC_NUM_MC_SAMPLES}")
    print(f"  • CRN Invariance:       {SMC_USE_CRN}")
    print(f"  • Smoothing Mode:       {SMC_SMOOTHING_MODE} (Song song toàn bộ M*N trên GPU)")
if is_lidar or is_rs_lidar:
    if found_lookahead_dir:
        print(f"  🔮 Lookahead P1:        {os.path.basename(found_lookahead_dir)} ({lookahead_count} prompts sẵn có trên Drive)")
    else:
        print(f"  🔮 Lookahead P1:        {LOOKAHEAD_TAG} (sẽ nạp từ Lookahead_samples/{LOOKAHEAD_TAG})")
if "SMC" in METHOD:
    print(f"  • Resample Freq:        Mỗi {RESAMPLE_FREQUENCY} bước (t ∈ [{RESAMPLE_T_START}, {RESAMPLE_T_END}])")
    print(f"  • Terminal Resample:    Mặc định bật cố định (bước 99 theo chuẩn tác giả Horvitz)")
    print(f"  • Feynman-Kac λ:        {FK_LMBDA}")
print(f"  • Tập prompt:           {PROMPT_MODE} ({MAX_PROMPT} prompts)")
print(f"  📁 Thư mục kết quả:      Target_samples/{RUN_NAME}")
print(f"  ⚡ Tiến độ sẵn có:       {existing_prompts}/{MAX_PROMPT} prompts trên Drive")
print("=" * 70)

if existing_prompts >= MAX_PROMPT and not OVERWRITE:
    print("  ✅ Thử nghiệm này đã hoàn tất đầy đủ 100%! Có thể chuyển ngay sang bước đánh giá.")
elif existing_prompts > 0:
    print(f"  ⚡ Đã có {existing_prompts} prompts. Cơ chế --resume sẽ tự động hoàn thành nốt các prompt còn lại.")


## 5. Kiểm Tra Tính Toàn Vẹn Pipeline (Integrity & Syntax Verification)
Biên dịch kiểm tra cú pháp Python của `LiDAR_sampling.py` và `fkd_class.py` trước khi thực thi để tránh lãng phí GPU Compute Units.


In [ ]:
import os, py_compile

# Kiểm tra tính toàn vẹn cú pháp của LiDAR_sampling.py và fkd_class.py
files_to_check = [
    f"{WORKDIR}/LiDAR_sampling.py",
    f"{WORKDIR}/fkd_diffusers/fkd_class.py"
]

all_passed = True
for filepath in files_to_check:
    if os.path.exists(filepath):
        try:
            py_compile.compile(filepath, doraise=True)
            print(f"✅ File {os.path.basename(filepath)} hoàn toàn chuẩn cú pháp!")
        except Exception as e:
            all_passed = False
            raise RuntimeError(f"❌ Lỗi cú pháp trong {filepath}: {e}")
    else:
        print(f"⚠️ Không tìm thấy file: {filepath}")

if all_passed:
    print("🚀 Pipeline hoàn toàn sẵn sàng thực thi an toàn 100%!")


## 6. Thực Thi Chạy Đơn Lẻ (Single Run Executor)
Chạy thử nghiệm theo cấu hình đã chọn ở Mục 4. Kết quả được lưu trực tiếp vào Google Drive theo từng prompt.


In [ ]:
import os, time

if EXECUTION_MODE != "SINGLE_RUN":
    print("=" * 70)
    print(f"⏩ [EXECUTION_MODE = '{EXECUTION_MODE}']: BỎ QUA MỤC 6 (Chạy Đơn Lẻ).")
    print("👉 Bạn đã chọn chế độ SWEEP_TABLE_8 ở Mục 4 nên Run đơn lẻ được bỏ qua hoàn toàn (0.00s)!")
    print("=" * 70)
else:
    os.environ["USE_TF"] = "0"
    os.environ["USE_TORCH"] = "1"
    %cd {WORKDIR}

    cmd = [
        "python", "LiDAR_sampling.py",
        f"--model_name={MODEL_NAME}",
        f"--num_particles={NUM_PARTICLES}",
        f"--num_inference_steps={NUM_INFERENCE_STEPS}",
        f"--eta={ETA}",
        f"--scale={SCALE}",
        f"--seed={SEED}",
        "--prompt_path=prompt_files/geneval_metadata.jsonl",
        f"--max_prompt={MAX_PROMPT}",
        f"--run_name={RUN_NAME}",
        "--metrics_to_compute=ImageReward#Clip-Score#HumanPreference",
        "--output_dir=Target_samples"
    ]

    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")

    if "RS-SMC" in METHOD:
        if NUM_PARTICLES > 1:
            cmd.extend([
                "--use_smc",
                "--use_smc_smoothing",
                f"--smc_sigma={SMC_SIGMA}",
                f"--smc_num_mc_samples={SMC_NUM_MC_SAMPLES}",
                f"--resample_frequency={RESAMPLE_FREQUENCY}",
                f"--FK_lmbda={FK_LMBDA}",
                f"--FK_resample_t_start={RESAMPLE_T_START}",
                f"--FK_resample_t_end={RESAMPLE_T_END}",
                f"--smc_terminal_resample={SMC_TERMINAL_RESAMPLE}",
                f"--smc_smoothing_mode={SMC_SMOOTHING_MODE}",
            ])
            if SMC_USE_CRN:
                cmd.append("--smc_use_crn=True")
            if SMC_ANNEAL_SIGMA:
                cmd.extend([
                    "--smc_anneal_sigma",
                    f"--smc_anneal_schedule={SMC_ANNEAL_SCHEDULE}",
                    f"--smc_sigma_min={SMC_SIGMA_MIN}"
                ])
        else:
            print("ℹ️ [NUM_PARTICLES=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")

    elif "Vanilla-SMC" in METHOD:
        if NUM_PARTICLES > 1:
            cmd.extend([
                "--use_smc",
                f"--resample_frequency={RESAMPLE_FREQUENCY}",
                f"--FK_lmbda={FK_LMBDA}",
                f"--FK_resample_t_start={RESAMPLE_T_START}",
                f"--FK_resample_t_end={RESAMPLE_T_END}",
                f"--smc_terminal_resample={SMC_TERMINAL_RESAMPLE}",
            ])
        else:
            print("ℹ️ [NUM_PARTICLES=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")

    elif METHOD == "BoN":
        pass

    if "LiDAR" in METHOD:
        cmd.extend([
            "--use_rag",
            f"--lookahead_path={LOOKAHEAD_PATH_ARG}",
            "--lmbda=5000",
            "--top_k=50"
        ])

    if SAVE_INDIVIDUAL_IMAGES:
        cmd.append("--save_individual_images")

    if not OVERWRITE:
        cmd.append("--resume")
    else:
        cmd.append("--overwrite")

    print("🚀 Đang khởi chạy thực nghiệm đơn lẻ:")
    for arg in cmd:
        print("   ", arg)
    print("-" * 70)

    t0 = time.time()
    !{" ".join(cmd)}
    elapsed = time.time() - t0
    print("-" * 70)
    print(f"✅ Hoàn thành thực nghiệm đơn lẻ trong {elapsed/60:.2f} phút!")
    print(f"📁 Toàn bộ kết quả đã được lưu trữ vĩnh viễn trên Drive tại: Target_samples/{RUN_NAME}")


## 7. Chạy Thử Nghiệm Sweep Bảng 8 (Table 8 Comparative Sweep Runner)
Tự động chạy quét toàn bộ dải hạt $N \in \{2, 4, 8\}$ so sánh trực tiếp giữa **Vanilla SMC** và **RS-SMC**. Tự động bỏ qua các run đã hoàn thành trên Google Drive.


In [ ]:
# @title ⚡ Tự Động Chạy Quét Dải Hạt Bảng 8 (Vanilla vs Static-RS vs Annealed-RS vs LiDAR)
import os, time, glob

if EXECUTION_MODE != "SWEEP_TABLE_8":
    print("=" * 70)
    print(f"⏩ [EXECUTION_MODE = '{EXECUTION_MODE}']: BỎ QUA MỤC 7 (Sweep Bảng 8).")
    print("👉 Bạn đã chọn chế độ SINGLE_RUN ở Mục 4 nên Sweep được bỏ qua hoàn toàn (0.00s)!")
    print("=" * 70)
else:

    INCLUDE_VANILLA_SMC = True # @param {type:"boolean"}
    INCLUDE_STATIC_RS_SMC = True # @param {type:"boolean"}
    INCLUDE_ANNEALED_RS_SMC = True # @param {type:"boolean"}
    INCLUDE_RS_SMC_LIDAR = False # @param {type:"boolean"}
    INCLUDE_RS_SMC_RS_LIDAR = False # @param {type:"boolean"}
    PARTICLES_SWEEP = "2,4,8" # @param {type:"string"}

    # Parse danh sách hạt cần sweep
    particles_list = [int(p.strip()) for p in PARTICLES_SWEEP.split(",") if p.strip()]

    # Thiết lập danh sách các cấu hình sweep cần chạy
    sweep_configs = []
    for n in particles_list:
        if INCLUDE_VANILLA_SMC:
            sweep_configs.append(("Vanilla-SMC", n))
        if INCLUDE_STATIC_RS_SMC:
            sweep_configs.append(("Static-RS-SMC", n))
        if INCLUDE_ANNEALED_RS_SMC:
            sweep_configs.append(("Annealed-RS-SMC", n))
        if INCLUDE_RS_SMC_LIDAR:
            sweep_configs.append(("RS-SMC + LiDAR", n))
        if INCLUDE_RS_SMC_RS_LIDAR:
            sweep_configs.append(("RS-SMC + RS-LiDAR", n))

    print("=" * 70)
    print(f"📋 DANH SÁCH {len(sweep_configs)} THỰC NGHIỆM SWEEP BẢNG 8 SẼ CHẠY:")
    for idx, (m_type, n_part) in enumerate(sweep_configs, 1):
        use_anneal_flag = (m_type == "Annealed-RS-SMC" or "RS-SMC" in m_type)
        method_key = "RS-SMC" if m_type in ["Static-RS-SMC", "Annealed-RS-SMC"] else m_type
        rname = build_systematic_run_name(
            model_tag=MODEL_TAG,
            method=method_key,
            num_particles=n_part,
            scale=SCALE,
            steps=NUM_INFERENCE_STEPS,
            resample_freq=RESAMPLE_FREQUENCY,
            fk_lmbda=FK_LMBDA,
            sigma=SMC_SIGMA,
            num_mc=SMC_NUM_MC_SAMPLES,
            use_crn=SMC_USE_CRN,
            use_anneal=use_anneal_flag,
            terminal_resample=SMC_TERMINAL_RESAMPLE,
            lk_sigma=LOOKAHEAD_SIGMA,
            lk_m=LOOKAHEAD_M,
            lk_crn=LOOKAHEAD_USE_CRN,
            seed=SEED,
            subset_tag=SUBSET_TAG
        )
        print(f"  {idx:02d}. [{m_type}] N={n_part} -> {rname}")
    print("=" * 70)

    for idx, (m_type, n_part) in enumerate(sweep_configs, 1):
        use_anneal_flag = (m_type == "Annealed-RS-SMC" or "RS-SMC" in m_type)
        method_key = "RS-SMC" if m_type in ["Static-RS-SMC", "Annealed-RS-SMC"] else m_type

        current_run_name = build_systematic_run_name(
            model_tag=MODEL_TAG,
            method=method_key,
            num_particles=n_part,
            scale=SCALE,
            steps=NUM_INFERENCE_STEPS,
            resample_freq=RESAMPLE_FREQUENCY,
            fk_lmbda=FK_LMBDA,
            sigma=SMC_SIGMA,
            num_mc=SMC_NUM_MC_SAMPLES,
            use_crn=SMC_USE_CRN,
            use_anneal=use_anneal_flag,
            terminal_resample=SMC_TERMINAL_RESAMPLE,
            lk_sigma=LOOKAHEAD_SIGMA,
            lk_m=LOOKAHEAD_M,
            lk_crn=LOOKAHEAD_USE_CRN,
            seed=SEED,
            subset_tag=SUBSET_TAG
        )
        current_dir = f"{WORKDIR}/Target_samples/{current_run_name}"
        existing = len(glob.glob(f"{current_dir}/[0-9]*/results.json")) if os.path.exists(current_dir) else 0

        print("")
        print(f"🔄 [{idx}/{len(sweep_configs)}] Đang xử lý: {current_run_name}")
        print(f"   Tiến độ hiện tại: {existing}/{MAX_PROMPT} prompts")

        if existing >= MAX_PROMPT and not OVERWRITE:
            print("   ⏩ Run đã hoàn tất 100%! Bỏ qua (Skip 0.00s).")
            continue

        # Xác định đường dẫn lookahead nếu method có LiDAR
        sweep_lk_arg = LOOKAHEAD_TAG
        if "RS-LiDAR" in m_type:
            crn_tag_lk = "_crn" if LOOKAHEAD_USE_CRN else ""
            sweep_lk_arg = f"RSLookahead_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{LOOKAHEAD_PARTICLES}_sig{LOOKAHEAD_SIGMA}_M{LOOKAHEAD_M}{crn_tag_lk}_seed{LOOKAHEAD_SEED}"
        elif "LiDAR" in m_type:
            sweep_lk_arg = f"Lookahead_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{LOOKAHEAD_PARTICLES}_seed{LOOKAHEAD_SEED}"

        sweep_cmd = [
            "python", "LiDAR_sampling.py",
            f"--model_name={MODEL_NAME}",
            f"--num_particles={n_part}",
            f"--num_inference_steps={NUM_INFERENCE_STEPS}",
            f"--eta={ETA}",
            f"--scale={SCALE}",
            f"--seed={SEED}",
            "--prompt_path=prompt_files/geneval_metadata.jsonl",
            f"--max_prompt={MAX_PROMPT}",
            f"--run_name={current_run_name}",
            "--metrics_to_compute=ImageReward#Clip-Score#HumanPreference",
            "--output_dir=Target_samples",
            "--resume"
        ]
        if PROMPT_INDICES_ARG:
            sweep_cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
        if SAVE_INDIVIDUAL_IMAGES:
            sweep_cmd.append("--save_individual_images")

        if "RS-SMC" in m_type or m_type in ["Static-RS-SMC", "Annealed-RS-SMC"]:
            if n_part > 1:
                sweep_cmd.extend([
                    "--use_smc",
                    "--use_smc_smoothing",
                    f"--smc_sigma={SMC_SIGMA}",
                    f"--smc_num_mc_samples={SMC_NUM_MC_SAMPLES}",
                    f"--resample_frequency={RESAMPLE_FREQUENCY}",
                    f"--FK_lmbda={FK_LMBDA}",
                    f"--FK_resample_t_start={RESAMPLE_T_START}",
                    f"--FK_resample_t_end={RESAMPLE_T_END}",
                    f"--smc_terminal_resample={SMC_TERMINAL_RESAMPLE}",
                    f"--smc_smoothing_mode={SMC_SMOOTHING_MODE}",
                ])
                if SMC_USE_CRN:
                    sweep_cmd.append("--smc_use_crn=True")
                if use_anneal_flag:
                    sweep_cmd.extend([
                        "--smc_anneal_sigma",
                        f"--smc_anneal_schedule={SMC_ANNEAL_SCHEDULE}",
                        f"--smc_sigma_min={SMC_SIGMA_MIN}"
                    ])
            else:
                print("   ℹ️ [N=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")
        elif "Vanilla-SMC" in m_type:
            if n_part > 1:
                sweep_cmd.extend([
                    "--use_smc",
                    f"--resample_frequency={RESAMPLE_FREQUENCY}",
                    f"--FK_lmbda={FK_LMBDA}",
                    f"--FK_resample_t_start={RESAMPLE_T_START}",
                    f"--FK_resample_t_end={RESAMPLE_T_END}",
                    f"--smc_terminal_resample={SMC_TERMINAL_RESAMPLE}",
                ])
            else:
                print("   ℹ️ [N=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")

        if "LiDAR" in m_type:
            sweep_cmd.extend([
                "--use_rag",
                f"--lookahead_path={sweep_lk_arg}",
                "--lmbda=5000",
                "--top_k=50"
            ])

        t_start = time.time()
        !{" ".join(sweep_cmd)}
        print(f"   ✅ Hoàn tất run trong {(time.time() - t_start)/60:.2f} phút.")

    print("")
    print("=" * 70)
    print("🎉 TOÀN BỘ CÁC THỰC NGHIỆM SWEEP BẢNG 8 ĐÃ HOÀN TẤT VÀ LƯU TRỮ TRÊN GOOGLE DRIVE!")


## 8. Chấm Điểm Benchmark GenEval Chuẩn Xác (Official Mask2Former Swin-S & HF CLIP)
Chấm điểm độ chính xác tạo ảnh chi tiết theo 6 nhóm nhiệm vụ chuẩn của GenEval (Single, Two-Object, Counting, Colors, Position, Color-Attr).


In [ ]:
# ==============================================================================
# 🎯 GENEVAL BENCHMARK EVALUATION (OFFICIAL MASK2FORMER SWIN-S & HF CLIP)
# ==============================================================================
RUN_GENEVAL = False # @param {type:"boolean"}
# Đặt True nếu bạn muốn chấm điểm Mask2Former Swin-S chính thức cho RUN_NAME.
# Đặt False (Mặc định khi sweep Table 8) để ưu tiên chạy Bảng 8 & xuất bảng tổng hợp tức thì.

if not RUN_GENEVAL:
    print("=" * 70)
    print("⏩ [RUN_GENEVAL=False]: Bỏ qua đánh giá GenEval Mask2Former.")
    print("👉 Chuyển thẳng sang Mục 9 để xuất Bảng Tổng Hợp Table 8 và Đồ Thị Scaling.")
    print("=" * 70)
else:
    # 🎯 GENEVAL BENCHMARK EVALUATION (OFFICIAL MASK2FORMER SWIN-S & HF CLIP)
    import os, glob, json, time, urllib.request
    import numpy as np, pandas as pd
    from PIL import Image
    import torch
    import torchvision.transforms.functional as TF
    from tqdm import tqdm
    from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation, CLIPModel, CLIPProcessor

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1. Tải metadata prompt chuẩn nếu chưa có
    metadata_path = f"{WORKDIR}/prompt_files/geneval_metadata.jsonl"
    if not os.path.exists(metadata_path):
        os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
        url = "https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl"
        urllib.request.urlretrieve(url, metadata_path)

    prompts_meta = []
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                prompts_meta.append(json.loads(line.strip()))
    print(f"📋 Đã nạp {len(prompts_meta)} prompt tiêu chuẩn GenEval.")

    # 2. Quét thư mục ảnh Target_samples theo đúng RUN_NAME / SCALE đã thiết lập
    if 'TARGET_DIR_OVERRIDE' in locals() and TARGET_DIR_OVERRIDE and os.path.exists(TARGET_DIR_OVERRIDE):
        target_dir = TARGET_DIR_OVERRIDE
    else:
        target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
        if not os.path.exists(target_dir):
            if 'DRIVE_DIR' in locals() and os.path.exists(f"{DRIVE_DIR}/Target_samples/{RUN_NAME}"):
                target_dir = f"{DRIVE_DIR}/Target_samples/{RUN_NAME}"
            else:
                # Tìm kiếm linh hoạt các biến thể (ví dụ Step5 <-> DPM5)
                alt_name = RUN_NAME.replace("Step5", "DPM5") if "Step5" in RUN_NAME else RUN_NAME.replace("DPM5", "Step5")
                if os.path.exists(f"{WORKDIR}/Target_samples/{alt_name}"):
                    target_dir = f"{WORKDIR}/Target_samples/{alt_name}"
                elif 'DRIVE_DIR' in locals() and os.path.exists(f"{DRIVE_DIR}/Target_samples/{alt_name}"):
                    target_dir = f"{DRIVE_DIR}/Target_samples/{alt_name}"

    print(f"📁 Thư mục đích: {target_dir}")

    # 3. Kiểm tra Checkpoint an toàn: Nếu đã có kết quả trước đó thì đọc ngay trong < 0.01s
    OVERWRITE_GENEVAL = False
    geneval_csv_path = f"{target_dir}/geneval_summary.csv"
    overall_geneval = 0.0

    if not OVERWRITE_GENEVAL:
        for candidate_csv in [f"{target_dir}/geneval_summary.csv", f"{target_dir}/geneval_mask2former_summary.csv"]:
            if os.path.exists(candidate_csv):
                try:
                    df_exist = pd.read_csv(candidate_csv)
                    ov_row = df_exist[df_exist.iloc[:, 0].astype(str).str.contains('OVERALL', case=False)]
                    if not ov_row.empty:
                        overall_geneval = float(ov_row.iloc[0, 2])
                        print(f"\n⚡ [CHECKPOINT CÓ SẴN] {RUN_NAME}: GenEval = {overall_geneval:.4f} -> Đã có kết quả, bỏ qua không cần chấm lại!")
                        print("   (Đặt OVERWRITE_GENEVAL = True ở đầu cell nếu muốn ép chấm lại từ đầu)\n")
                        print("="*70)
                        print(df_exist.to_string(index=False))
                        print("="*70)
                        break
                except Exception:
                    pass

    if overall_geneval == 0.0 or OVERWRITE_GENEVAL:
        # 4. Nạp mô hình Mask2Former Swin-S chuẩn COCO và OpenCLIP ViT-L-14
        print("\n📦 Đang nạp mô hình Mask2Former Swin-S COCO (facebook/mask2former-swin-small-coco-instance)...")
        print("   (Chính xác mô hình chuẩn của bài báo GenEval, mAP ~52%)")
        mask2former_id = "facebook/mask2former-swin-small-coco-instance"
        image_processor = AutoImageProcessor.from_pretrained(mask2former_id)
        image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}  # Chuẩn bài báo GenEval / MMDetection COCO gốc (800px)
        detector = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_id).to(device).eval()
        id2label = detector.config.id2label

        # Danh sách 80 lớp COCO chuẩn bài báo GenEval (object_names.txt)
        COCO_CLASSES_80 = [
            "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
            "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
            "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
            "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
            "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
            "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
            "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "computer mouse", "tv remote",
            "computer keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book",
            "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
        ]

        # 5. Nạp mô hình OpenCLIP ViT-L-14 (Chuẩn bài báo GenEval gốc 100%)
        print("\n📦 Đang nạp mô hình OpenCLIP ViT-L-14 (openai weights) cho kiểm tra màu sắc (colors & color_attr)...\n")
        try:
            import open_clip
            clip_arch = "ViT-L-14"
            clip_model, _, clip_transform = open_clip.create_model_and_transforms(clip_arch, pretrained="openai", device=device)
            clip_tokenizer = open_clip.get_tokenizer(clip_arch)
            clip_model.eval()
            use_open_clip = True
            print("✅ Đã nạp thành công OpenCLIP ViT-L-14 chuẩn gốc!")
        except Exception as e:
            print(f"⚠️ Không nạp được open_clip ({e}), chuyển sang dự phòng Hugging Face CLIP ViT-L/14...")
            from transformers import CLIPModel, CLIPProcessor
            clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
            clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
            clip_transform = lambda img: clip_processor(images=img, return_tensors="pt")["pixel_values"][0]
            clip_tokenizer = None
            use_open_clip = False

        COLORS = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white']
        COLOR_CLASSIFIERS = {}

        def get_color_classifier(classname):
            """Xây dựng và cache zero-shot color classifier chuẩn GenEval cho từng classname"""
            if classname not in COLOR_CLASSIFIERS:
                c_name = classname if classname else "object"
                templates = [
                    f"a photo of a {{c}} {c_name}",
                    f"a photo of a {{c}}-colored {c_name}",
                    f"a photo of a {{c}} object"
                ]
                with torch.inference_mode():
                    if use_open_clip:
                        zeroshot_weights = []
                        for c in COLORS:
                            prompts = [t.format(c=c) for t in templates]
                            tokens = clip_tokenizer(prompts).to(device)
                            text_embeddings = clip_model.encode_text(tokens)
                            text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)
                            class_embedding = text_embeddings.mean(dim=0)
                            class_embedding = class_embedding / class_embedding.norm(dim=-1, keepdim=True)
                            zeroshot_weights.append(class_embedding)
                        COLOR_CLASSIFIERS[classname] = torch.stack(zeroshot_weights, dim=1)  # (D, 10)
                    else:
                        prompts = [t.format(c=c) for c in COLORS for t in templates]
                        inputs = clip_processor(text=prompts, return_tensors="pt", padding=True).to(device)
                        text_out = clip_model.get_text_features(**inputs)
                        text_out = text_out / text_out.norm(dim=-1, keepdim=True)
                        COLOR_CLASSIFIERS[classname] = text_out.view(len(COLORS), 3, -1).mean(dim=1).T  # (D, 10)
            return COLOR_CLASSIFIERS[classname]

        def color_classification(image, objects, classname):
            """Phân loại màu sắc chuẩn 100% official GenEval: Ghép mask nền xám #999 + CLIP ViT-L-14"""
            if not objects:
                return []
            clf = get_color_classifier(classname)
            blank = Image.new("RGB", image.size, color="#999")
            crop_tensors = []
            for obj in objects:
                box, mask = obj[0], obj[1] if len(obj) > 1 else None
                x1, y1, x2, y2 = int(max(0, box[0])), int(max(0, box[1])), int(min(image.width, box[2])), int(min(image.height, box[3]))
                if (x2 - x1 <= 12 or y2 - y1 <= 12):
                    crop_tensors.append(clip_transform(Image.new("RGB", (224, 224), color="#999")))
                    continue
                if mask is not None:
                    comp = Image.composite(image, blank, Image.fromarray((mask > 0).astype(np.uint8) * 255))
                else:
                    comp = image
                crop = comp.crop((x1, y1, x2, y2))
                crop_tensors.append(clip_transform(crop))

            crop_batch = torch.stack(crop_tensors).to(device)
            with torch.inference_mode():
                if use_open_clip:
                    img_features = clip_model.encode_image(crop_batch)
                else:
                    img_features = clip_model.get_image_features(pixel_values=crop_batch)
                img_features = img_features / img_features.norm(dim=-1, keepdim=True)
                logits = 100.0 * (img_features @ clf)
                pred_indices = logits.argmax(dim=-1).cpu().tolist()
                return [COLORS[idx] for idx in pred_indices]

        def relative_position(obj_a, obj_b):
            """Tính vị trí tương quan của A đối với B chuẩn 100% bài báo GenEval gốc (evaluate_images.py:relative_position)"""
            box_a = obj_a[0] if isinstance(obj_a, (tuple, list)) else obj_a
            box_b = obj_b[0] if isinstance(obj_b, (tuple, list)) else obj_b
            boxes = np.array([box_a[:4], box_b[:4]])[:, :4].reshape(2, 2, 2)
            center_a, center_b = boxes.mean(axis=-2)
            dim_a, dim_b = np.abs(np.diff(boxes, axis=-2))[..., 0, :]
            offset = center_a - center_b
            revised_offset = np.maximum(np.abs(offset) - 0.1 * (dim_a + dim_b), 0) * np.sign(offset)
            if np.all(np.abs(revised_offset) < 1e-3):
                return set()
            norm_offset = np.linalg.norm(offset)
            if norm_offset <= 1e-5:
                return set()
            dx, dy = revised_offset / norm_offset
            relations = set()
            if dx < -0.5: relations.add("left of")
            if dx > 0.5: relations.add("right of")
            if dy < -0.5: relations.add("above")
            if dy > 0.5: relations.add("below")
            return relations

        def evaluate_single_image(image, objects, metadata):
            """Đánh giá 1 ảnh chuẩn 100% theo official GenEval (evaluate_images.py:evaluate)"""
            correct = True
            matched_groups = []
            # 1. Kiểm tra các vật thể được yêu cầu (include)
            for req in metadata.get('include', []):
                classname = req['class'].lower()
                matched = True
                found_objects = objects.get(classname, [])[:req['count']]
                if len(found_objects) < req['count']:
                    correct = matched = False
                else:
                    if 'color' in req:
                        colors = color_classification(image, found_objects, classname)
                        if colors.count(req['color'].lower()) < req['count']:
                            correct = matched = False
                    if 'position' in req and matched:
                        expected_rel, target_group = req['position']
                        if target_group >= len(matched_groups) or matched_groups[target_group] is None:
                            correct = matched = False
                        else:
                            for obj in found_objects:
                                for target_obj in matched_groups[target_group]:
                                    true_rels = relative_position(obj, target_obj)
                                    if expected_rel not in true_rels:
                                        correct = matched = False
                                        break
                                if not matched:
                                    break
                if matched:
                    matched_groups.append(found_objects)
                else:
                    matched_groups.append(None)
            # 2. Kiểm tra các vật thể bị loại trừ (exclude - đặc biệt quan trọng cho task counting)
            for req in metadata.get('exclude', []):
                classname = req['class'].lower()
                if len(objects.get(classname, [])) >= req['count']:
                    correct = False
            return correct

        task_results = {'single_object': [], 'two_object': [], 'counting': [], 'colors': [], 'position': [], 'color_attr': []}
        scored_prompts_count = 0
        err_count = 0

        print("\n" + "="*70)
        print("🚀 Bắt đầu chấm điểm GenEval (Mask2Former Swin-S)...")
        print("="*70)

        # VÒNG LẶP CHẠY TRỰC TIẾP TỪ 0 ĐẾN 552: TQDM HIỆN NGAY TỨC THÌ (0s DELAY)
        for p_idx in tqdm(range(len(prompts_meta)), desc="Chấm điểm GenEval (Mask2Former)"):
            meta = prompts_meta[p_idx]
            tag = meta.get('tag', 'single_object')
            if tag not in task_results:
                continue

            # Tìm ảnh trực tiếp tại thư mục prompt p_idx (hỗ trợ cả int và 5-digit zero-pad)
            p_dir = f"{target_dir}/{p_idx}"
            if not os.path.exists(p_dir):
                p_dir_alt = f"{target_dir}/{p_idx:05d}"
                if os.path.exists(p_dir_alt):
                    p_dir = p_dir_alt
                else:
                    continue

            imgs = sorted(glob.glob(f"{p_dir}/samples/*.png")) or \
                   sorted(glob.glob(f"{p_dir}/best_of_n_samples/*.png")) or \
                   sorted([img for img in glob.glob(f"{p_dir}/*.png") if not img.endswith("grid.png")])
            if not imgs:
                continue

            scored_prompts_count += 1
            prompt_scores = []
            for img_path in imgs:
                try:
                    img = Image.open(img_path).convert('RGB')
                    # Cấu hình chuẩn bài báo GenEval gốc: Cạnh ngắn luôn là 800px (chuẩn MMDetection COCO)
                    image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}
                    inputs = image_processor(images=img, return_tensors="pt").to(device)
                    with torch.inference_mode():
                        outputs = detector(**inputs)

                    # Ngưỡng tin cậy chuẩn bài báo GenEval gốc: 0.9 cho counting, 0.3 cho tất cả các task khác
                    conf_thr = 0.9 if tag == 'counting' else 0.3
                    results = image_processor.post_process_instance_segmentation(
                        outputs, target_sizes=[img.size[::-1]], threshold=conf_thr
                    )[0]
                    segmentation = results["segmentation"].detach().cpu().numpy()
                    segments_info = results["segments_info"]

                    # Sắp xếp các vật thể theo confidence giảm dần (chuẩn bài báo GenEval gốc)
                    segments_info = sorted(segments_info, key=lambda s: s.get("score", 0.0), reverse=True)

                    detected = {}
                    for seg in segments_info:
                        lbl_id = seg["label_id"]
                        c_name = COCO_CLASSES_80[lbl_id].lower() if (0 <= lbl_id < len(COCO_CLASSES_80)) else id2label.get(lbl_id, "unknown").lower()
                        mask = (segmentation == seg["id"])
                        y_indices, x_indices = np.where(mask)
                        if len(x_indices) > 0 and len(y_indices) > 0:
                            x1, x2 = float(np.min(x_indices)), float(np.max(x_indices))
                            y1, y2 = float(np.min(y_indices)), float(np.max(y_indices))
                            box = [x1, y1, x2, y2, seg.get("score", 1.0)]

                            # Giới hạn MAX_OBJECTS = 16 mỗi lớp (chuẩn official GenEval)
                            if len(detected.setdefault(c_name, [])) < 16:
                                detected[c_name].append((box, mask))

                    success = evaluate_single_image(img, detected, meta)
                    prompt_scores.append(1.0 if success else 0.0)
                except Exception as e:
                    err_count += 1
                    if err_count <= 3:
                        print(f"⚠️ [Lỗi ảnh {img_path}]: {e}")

            if prompt_scores:
                task_results[tag].append(np.mean(prompt_scores))

        if scored_prompts_count > 0:
            summary_rows = []
            all_means = []
            for t_name, scores in task_results.items():
                mean_val = np.mean(scores) if scores else 0.0
                if scores: all_means.append(mean_val)
                summary_rows.append({'Nhiệm Vụ (Task)': t_name, 'Số Lượng Prompt': len(scores), 'Độ Chính Xác (Accuracy ↑)': f"{mean_val:.4f}"})

            overall_geneval = float(np.mean(all_means)) if all_means else 0.0
            summary_rows.append({'Nhiệm Vụ (Task)': '🔥 OVERALL GENEVAL BENCHMARK', 'Số Lượng Prompt': sum(len(s) for s in task_results.values()), 'Độ Chính Xác (Accuracy ↑)': f"{overall_geneval:.4f}"})
            df_geneval = pd.DataFrame(summary_rows)
            print('\n' + '='*70)
            print('📊 BẢNG TỔNG HỢP ĐIỂM SỐ GENEVAL BENCHMARK (MASK2FORMER CHUẨN BÀI BÁO)')
            print('='*70)
            print(df_geneval.to_string(index=False))
            print('='*70)
            df_geneval.to_csv(geneval_csv_path, index=False)
            df_geneval.to_csv(f"{target_dir}/geneval_mask2former_summary.csv", index=False)

            # Đánh dấu hoàn thành bằng file marker tương thích 100% với Rescore notebook
            with open(f"{target_dir}/.mask2former_rescore_done", 'w', encoding='utf-8') as f:
                json.dump({
                    "completed_at": time.ctime(),
                    "evaluator": "facebook/mask2former-swin-small-coco-instance",
                    "geneval_score": overall_geneval,
                    "prompts_evaluated": scored_prompts_count
                }, f, indent=2)

            if 'DRIVE_DIR' in locals() and DRIVE_DIR and os.path.exists(DRIVE_DIR):
                drive_exp_dir = f"{DRIVE_DIR}/Target_samples/{os.path.basename(target_dir)}"
                os.makedirs(drive_exp_dir, exist_ok=True)
                df_geneval.to_csv(f"{drive_exp_dir}/geneval_summary.csv", index=False)
                df_geneval.to_csv(f"{drive_exp_dir}/geneval_mask2former_summary.csv", index=False)
                with open(f"{drive_exp_dir}/.mask2former_rescore_done", 'w', encoding='utf-8') as f:
                    json.dump({
                        "completed_at": time.ctime(),
                        "evaluator": "facebook/mask2former-swin-small-coco-instance",
                        "geneval_score": overall_geneval,
                        "prompts_evaluated": scored_prompts_count
                    }, f, indent=2)
        else:
            print('\n❌ Không tìm thấy ảnh hợp lệ trong thư mục đích để tính điểm GenEval.')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 9. Tổng Kết Bảng So Sánh Chuẩn Bảng 8 & Đồ Thị (Publication-Grade Table 8)
Trực quan hóa và lập bảng so sánh ImageReward giữa **Vanilla SMC** và **RS-SMC** theo các mức hạt $N \in \{1, 2, 4, 8\}$ đối chiếu trực tiếp với kết quả bài báo gốc.


In [ ]:
# @title 📊 BƯỚC 9: BẢNG TỔNG HỢP SO SÁNH CHUẨN BẢNG 8 BÀI BÁO GỐC (THEO TỪNG GIÁ TRỊ N)
import pandas as pd, glob, os, json
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# ------------------------------------------------------------------------------
# 1. HÀM TRÍCH XUẤT CHỈ SỐ TOÀN DIỆN (IMAGEREWARD, CLIP, HPSv2, GENEVAL)
# ------------------------------------------------------------------------------
def read_run_metrics(folder_name):
    folder_path = f"{WORKDIR}/Target_samples/{folder_name}"
    if not os.path.exists(folder_path):
        # Fallback check alternate terminal resample folder if available
        alt_folder = folder_name.replace("_noTerminal", "") if "_noTerminal" in folder_name else folder_name.replace("_freq", "_noTerminal_freq")
        if os.path.exists(f"{WORKDIR}/Target_samples/{alt_folder}"):
            folder_path = f"{WORKDIR}/Target_samples/{alt_folder}"
        else:
            return None
    
    ir, clip, hps, geneval = None, None, None, None
    count = len(glob.glob(f"{folder_path}/[0-9]*"))

    # (A) Đọc nhanh từ final_metrics.json nếu run đã hoàn tất
    for fm_name in ["final_metrics.json", "final_metrics_shard_0.json"]:
        fm_path = os.path.join(folder_path, fm_name)
        if os.path.exists(fm_path):
            try:
                with open(fm_path, "r", encoding="utf-8") as f:
                    fm = json.load(f)
                ir = fm.get("ImageReward", {}).get("mean", None)
                clip = fm.get("Clip-Score", {}).get("mean", None)
                hps = fm.get("HumanPreference", {}).get("mean", None)
                break
            except Exception:
                pass

    # (B) Dự phòng: Quét và tính trung bình trực tiếp từ các file results.json nếu chưa có final_metrics
    if ir is None:
        r_files = glob.glob(f"{folder_path}/[0-9]*/results.json")
        if r_files:
            ir_list, clip_list, hps_list = [], [], []
            for rf in r_files:
                try:
                    with open(rf, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if "ImageReward" in data and "mean" in data["ImageReward"] and data["ImageReward"]["mean"] is not None:
                        ir_list.append(data["ImageReward"]["mean"])
                    if "Clip-Score" in data and "mean" in data["Clip-Score"] and data["Clip-Score"]["mean"] is not None:
                        clip_list.append(data["Clip-Score"]["mean"])
                    if "HumanPreference" in data and "mean" in data["HumanPreference"] and data["HumanPreference"]["mean"] is not None:
                        hps_list.append(data["HumanPreference"]["mean"])
                except Exception:
                    pass
            ir = (sum(ir_list) / len(ir_list)) if ir_list else None
            clip = (sum(clip_list) / len(clip_list)) if clip_list else None
            hps = (sum(hps_list) / len(hps_list)) if hps_list else None
            count = len(r_files)

    # (C) Đọc điểm GenEval nếu đã có file geneval_summary.csv hoặc marker
    for g_csv in ["geneval_summary.csv", "geneval_mask2former_summary.csv"]:
        gp = os.path.join(folder_path, g_csv)
        if os.path.exists(gp):
            try:
                df_g = pd.read_csv(gp)
                ov_row = df_g[df_g.iloc[:, 0].astype(str).str.contains("OVERALL", case=False)]
                if not ov_row.empty:
                    geneval = float(ov_row.iloc[0, 2])
                    break
            except Exception:
                pass
    if geneval is None:
        marker_p = os.path.join(folder_path, ".mask2former_rescore_done")
        if os.path.exists(marker_p):
            try:
                with open(marker_p, "r", encoding="utf-8") as f:
                    geneval = json.load(f).get("geneval_score", None)
            except Exception:
                pass

    if ir is None and geneval is None:
        return None
    return {"ir": ir, "clip": clip, "hps": hps, "geneval": geneval, "count": count}

# ------------------------------------------------------------------------------
# 2. XUẤT KẾT QUẢ RIÊNG CHO CHẾ ĐỘ SINGLE_RUN (NẾU ĐƯỢC CHỌN)
# ------------------------------------------------------------------------------
if EXECUTION_MODE == "SINGLE_RUN":
    m_single = read_run_metrics(RUN_NAME)
    print("=" * 80)
    print(f"🎯 KẾT QUẢ THỰC NGHIỆM ĐƠN LẺ ({RUN_NAME}):")
    print("=" * 80)
    if m_single:
        print(f"  • Số prompt hoàn tất:   {m_single['count']}/{MAX_PROMPT}")
        print(f"  • ImageReward:          {m_single['ir']:.4f}" if m_single['ir'] is not None else "  • ImageReward:          Chưa có")
        print(f"  • CLIP-Score:           {m_single['clip']:.4f}" if m_single['clip'] is not None else "  • CLIP-Score:           Chưa có")
        print(f"  • HumanPreference:      {m_single['hps']:.4f}" if m_single['hps'] is not None else "  • HumanPreference:      Chưa có")
        print(f"  • GenEval (Mask2Former): {m_single['geneval']:.4f}" if m_single['geneval'] is not None else "  • GenEval (Mask2Former): Chưa chấm (Bật RUN_GENEVAL=True ở Mục 8 để chấm)")
    else:
        print(f"  ⚠️ Chưa tìm thấy dữ liệu kết quả trong Target_samples/{RUN_NAME}")
    print("=" * 80)
    print("")

# ------------------------------------------------------------------------------
# 3. BẢNG THAM CHIẾU BÀI BÁO GỐC (TABLE 8 & TABLE 2 BENCHMARKS)
# ------------------------------------------------------------------------------
# Bảng đối chiếu chính thức từ Table 8 bài báo LiDAR (ICML 2026)
paper_ref = {
    1: {"vanilla_smc": -0.001, "lidar": 0.384, "bon": -0.001, "smc_bon": -0.001},
    2: {"vanilla_smc": 0.267,  "lidar": 0.715, "bon": 0.414,  "smc_bon": 0.336},
    4: {"vanilla_smc": 0.568,  "lidar": 1.007, "bon": 0.739,  "smc_bon": 0.692},
    8: {"vanilla_smc": 0.864,  "lidar": 1.122, "bon": 0.948,  "smc_bon": 1.015}
}

# Lấy danh sách hạt N: Kết hợp dải chuẩn [1, 2, 4, 8] và dải do người dùng chỉ định
sweep_n_list = [int(p.strip()) for p in PARTICLES_SWEEP.split(",") if p.strip()]
all_n_values = sorted(list(set([1, 2, 4, 8] + sweep_n_list)))

master_rows = []
table8_clean_rows = []
measured_curves = {"vanilla_smc": {}, "static_rs_smc": {}, "annealed_rs_smc": {}, "rs_smc_lidar": {}, "rs_smc_rslidar": {}}

for n in all_n_values:
    # (1) Vanilla SMC
    smc_name = build_systematic_run_name(MODEL_TAG, "Vanilla-SMC", n, SCALE, NUM_INFERENCE_STEPS,
                                         RESAMPLE_FREQUENCY, FK_LMBDA, SMC_SIGMA, SMC_NUM_MC_SAMPLES,
                                         SMC_USE_CRN, False, SMC_TERMINAL_RESAMPLE, SEED, SUBSET_TAG)
    m_smc = read_run_metrics(smc_name)
    ir_smc = m_smc["ir"] if m_smc else None
    clip_smc = m_smc["clip"] if m_smc else None
    hps_smc = m_smc["hps"] if m_smc else None
    g_smc = m_smc["geneval"] if m_smc else None
    if ir_smc is not None:
        measured_curves["vanilla_smc"][n] = ir_smc

    # (2) Static RS-SMC (Fixed sigma)
    static_name = build_systematic_run_name(MODEL_TAG, "RS-SMC", n, SCALE, NUM_INFERENCE_STEPS,
                                            RESAMPLE_FREQUENCY, FK_LMBDA, SMC_SIGMA, SMC_NUM_MC_SAMPLES,
                                            SMC_USE_CRN, False, SMC_TERMINAL_RESAMPLE, SEED, SUBSET_TAG)
    m_static = read_run_metrics(static_name)
    ir_static = m_static["ir"] if m_static else None
    clip_static = m_static["clip"] if m_static else None
    hps_static = m_static["hps"] if m_static else None
    g_static = m_static["geneval"] if m_static else None
    if ir_static is not None:
        measured_curves["static_rs_smc"][n] = ir_static

    # (3) Annealed RS-SMC (Adaptive sigma)
    anneal_name = build_systematic_run_name(MODEL_TAG, "RS-SMC", n, SCALE, NUM_INFERENCE_STEPS,
                                            RESAMPLE_FREQUENCY, FK_LMBDA, SMC_SIGMA, SMC_NUM_MC_SAMPLES,
                                            SMC_USE_CRN, True, SMC_TERMINAL_RESAMPLE, SEED, SUBSET_TAG)
    m_anneal = read_run_metrics(anneal_name)
    ir_anneal = m_anneal["ir"] if m_anneal else None
    clip_anneal = m_anneal["clip"] if m_anneal else None
    hps_anneal = m_anneal["hps"] if m_anneal else None
    g_anneal = m_anneal["geneval"] if m_anneal else None
    if ir_anneal is not None:
        measured_curves["annealed_rs_smc"][n] = ir_anneal

    # Tính toán mức chênh lệch (Delta Gains)
    delta_anneal_vs_smc = f"+{ir_anneal - ir_smc:.3f}" if (ir_anneal is not None and ir_smc is not None) else "—"
    delta_anneal_vs_static = f"+{ir_anneal - ir_static:.3f}" if (ir_anneal is not None and ir_static is not None) else "—"

    paper_smc_ref = paper_ref.get(n, {}).get("vanilla_smc", None)
    paper_lidar_ref = paper_ref.get(n, {}).get("lidar", None)

    # BẢNG 1: BẢNG TỔNG HỢP TOÀN DIỆN CÁC CHỈ SỐ (MỖI DÒNG LÀ 1 GIÁ TRỊ N)
    master_rows.append({
        "N (Particles)": n,
        "ImageReward (Vanilla SMC)": f"{ir_smc:.3f}" if ir_smc is not None else "Chưa chạy",
        "ImageReward (Static RS)": f"{ir_static:.3f}" if ir_static is not None else "Chưa chạy",
        "ImageReward (Annealed RS)": f"{ir_anneal:.3f}" if ir_anneal is not None else "Chưa chạy",
        "Δ IR Gain (Anneal vs SMC)": delta_anneal_vs_smc,
        "CLIP-Score (Vanilla)": f"{clip_smc:.3f}" if clip_smc is not None else "Chưa chạy",
        "CLIP-Score (Annealed)": f"{clip_anneal:.3f}" if clip_anneal is not None else "Chưa chạy",
        "HPSv2 (Vanilla)": f"{hps_smc:.3f}" if hps_smc is not None else "Chưa chạy",
        "HPSv2 (Annealed)": f"{hps_anneal:.3f}" if hps_anneal is not None else "Chưa chạy",
        "GenEval (Vanilla)": f"{g_smc:.4f}" if g_smc is not None else "—",
        "GenEval (Annealed)": f"{g_anneal:.4f}" if g_anneal is not None else "—",
        "Paper SMC Ref (IR)": f"{paper_smc_ref:.3f}" if paper_smc_ref is not None else "—",
        "Paper LiDAR Ref (IR)": f"{paper_lidar_ref:.3f}" if paper_lidar_ref is not None else "—"
    })

    paper_bon_ref = paper_ref.get(n, {}).get("bon", None)
    # BẢNG 2: BẢNG CHUYÊN BIỆT ĐỐI CHIẾU CHUẨN 1-1 BẢNG 8 BÀI BÁO (IMAGE REWARD)
    table8_clean_rows.append({
        "N (Particles)": n,
        "Paper SMC Baseline": f"{paper_smc_ref:.3f}" if paper_smc_ref is not None else "—",
        "Paper BoN Baseline": f"{paper_bon_ref:.3f}" if paper_bon_ref is not None else "—",
        "Measured Vanilla SMC": f"{ir_smc:.3f}" if ir_smc is not None else "Chưa chạy",
        "Measured Static RS-SMC": f"{ir_static:.3f}" if ir_static is not None else "Chưa chạy",
        "Measured Annealed RS-SMC": f"{ir_anneal:.3f}" if ir_anneal is not None else "Chưa chạy",
        "RS-SMC + LiDAR": f"{ir_rs_lidar:.3f}" if ir_rs_lidar is not None else "—",
        "RS-SMC + RS-LiDAR": f"{ir_rs_rslidar:.3f}" if ir_rs_rslidar is not None else "—",
        "Δ Anneal vs Vanilla": delta_anneal_vs_smc,
        "Δ Anneal vs Static": delta_anneal_vs_static,
        "Paper LiDAR Ref": f"{paper_lidar_ref:.3f}" if paper_lidar_ref is not None else "—"
    })

df_master = pd.DataFrame(master_rows)
df_table8 = pd.DataFrame(table8_clean_rows)

# Lưu kết quả vĩnh viễn trên Google Drive
os.makedirs(f"{WORKDIR}/Target_samples", exist_ok=True)
csv_path = f"{WORKDIR}/Target_samples/Table8_Comprehensive_Results.csv"
md_path = f"{WORKDIR}/Target_samples/Table8_Comprehensive_Results.md"
df_master.to_csv(csv_path, index=False)
with open(md_path, "w", encoding="utf-8") as f:
    f.write(f"# 📊 BẢNG TỔNG HỢP SO SÁNH TABLE 8 ({MODEL_TAG} - {PROMPT_MODE})\n\n")
    f.write("### 1. Bảng Tổng Hợp Toàn Bộ Chỉ Số (ImageReward, CLIP-Score, HPSv2, GenEval)\n\n")
    f.write(df_master.to_markdown(index=False))
    f.write("\n\n### 2. Bảng Đối Chiếu Chuẩn 1-1 Table 8 Bài Báo Gốc (ImageReward)\n\n")
    f.write(df_table8.to_markdown(index=False))
    f.write("\n")

# ------------------------------------------------------------------------------
# 4. HIỂN THỊ TRỰC QUAN GIAO DIỆN (EASY-TO-READ DISPLAY)
# ------------------------------------------------------------------------------
print("=" * 100)
print(f"📊 BẢNG 1: TỔNG HỢP TOÀN DIỆN CÁC CHỈ SỐ THEO TỪNG GIÁ TRỊ N ({MODEL_TAG} - {PROMPT_MODE}):")
print("   (Mỗi dòng là 1 giá trị N, các cột biểu thị từng chỉ số độc lập)")
print("=" * 100)
print(df_master.to_string(index=False))
print("=" * 100)
print("")
print("=" * 100)
print(f"🎯 BẢNG 2: ĐỐI CHIẾU CHUẨN 1-1 VỚI TABLE 8 BÀI BÁO GỐC (IMAGE REWARD):")
print("=" * 100)
print(df_table8.to_string(index=False))
print("=" * 100)
print(f"💾 Toàn bộ bảng đã được tự động lưu vào Drive tại:\n   📁 {csv_path}\n   📄 {md_path}")

# ------------------------------------------------------------------------------
# 5. VẼ ĐỒ THỊ SCALING CURVE (IMAGEREWARD VS. N)
# ------------------------------------------------------------------------------
if any(measured_curves.values()):
    plt.figure(figsize=(10.5, 6), dpi=160)
    sns.set_style("whitegrid")
    
    # Đường tham chiếu chuẩn bài báo gốc (Paper Table 8)
    paper_x = [1, 2, 4, 8]
    paper_y_smc = [paper_ref[x]["vanilla_smc"] for x in paper_x]
    paper_y_bon = [paper_ref[x]["bon"] for x in paper_x]
    paper_y_lidar = [paper_ref[x]["lidar"] for x in paper_x]
    plt.plot(paper_x, paper_y_smc, "k--s", linewidth=1.8, markersize=6, label="Paper Vanilla SMC (Table 8)", alpha=0.7)
    plt.plot(paper_x, paper_y_bon, "g:o", linewidth=1.8, markersize=6, label="Paper BoN (Table 8)", alpha=0.7)
    plt.plot(paper_x, paper_y_lidar, "c-.d", linewidth=1.8, markersize=6, label="Paper SMC + LiDAR (Table 8)", alpha=0.7)

    if measured_curves["vanilla_smc"]:
        x_v = sorted(measured_curves["vanilla_smc"].keys())
        y_v = [measured_curves["vanilla_smc"][k] for k in x_v]
        plt.plot(x_v, y_v, "b-o", linewidth=2.2, markersize=7, label="Measured Vanilla SMC")

    if measured_curves["static_rs_smc"]:
        x_st = sorted(measured_curves["static_rs_smc"].keys())
        y_st = [measured_curves["static_rs_smc"][k] for k in x_st]
        plt.plot(x_st, y_st, "m-^", linewidth=2.2, markersize=8, label=f"Measured Static RS-SMC (σ={SMC_SIGMA})")

    if measured_curves["annealed_rs_smc"]:
        x_an = sorted(measured_curves["annealed_rs_smc"].keys())
        y_an = [measured_curves["annealed_rs_smc"][k] for k in x_an]
        plt.plot(x_an, y_an, "r-*", linewidth=2.8, markersize=10, label=f"Measured Annealed RS-SMC (Ours, σ_base={SMC_SIGMA})")

    if measured_curves.get("rs_smc_lidar"):
        x_rl = sorted(measured_curves["rs_smc_lidar"].keys())
        y_rl = [measured_curves["rs_smc_lidar"][k] for k in x_rl]
        plt.plot(x_rl, y_rl, "m-d", linewidth=2.2, markersize=7, label="Measured RS-SMC + LiDAR")

    if measured_curves.get("rs_smc_rslidar"):
        x_rsl = sorted(measured_curves["rs_smc_rslidar"].keys())
        y_rsl = [measured_curves["rs_smc_rslidar"][k] for k in x_rsl]
        plt.plot(x_rsl, y_rsl, "c-p", linewidth=2.5, markersize=8, label="Measured RS-SMC + RS-LiDAR")

    plt.axhline(0.384, color="green", linestyle=":", linewidth=2, label="LiDAR N=4 Baseline (+0.384)")
    plt.title(f"Table 8 Scaling Curve: ImageReward vs. Particles N ({MODEL_TAG})", fontsize=13, fontweight="bold")
    plt.xlabel("Number of Particles N", fontsize=11, fontweight="bold")
    plt.ylabel("ImageReward Score", fontsize=11, fontweight="bold")
    plt.xticks(all_n_values)
    plt.legend(frameon=True, loc="lower right", fontsize=10)
    plt.tight_layout()
    
    plot_path = f"{WORKDIR}/Target_samples/Table8_Scaling_Curve_{MODEL_TAG}.png"
    plt.savefig(plot_path, dpi=200)
    print(f"📈 Đồ thị scaling đã được lưu tại: {plot_path}")
    plt.show()


## 10. Trực Quan Hóa Lưới Ảnh Mẫu Đã Sinh Trên Google Drive
Hiển thị trực quan các ảnh đã sinh của run hiện tại để kiểm tra chất lượng hình ảnh.


In [ ]:
import glob
from PIL import Image
from IPython.display import display

current_target = f"{WORKDIR}/Target_samples/{RUN_NAME}"
sample_images = sorted(glob.glob(f"{current_target}/[0-9]*/samples/*.png"))[:6]

if sample_images:
    print(f"🖼️ Hiển thị {len(sample_images)} ảnh mẫu từ {RUN_NAME}:")
    for img_path in sample_images:
        prompt_id = img_path.split(os.sep)[-3]
        print(f"• Prompt {prompt_id}: {img_path}")
        display(Image.open(img_path).resize((384, 384)))
else:
    print(f"ℹ️ Chưa có ảnh nào trong {current_target}")


## 11. (Tùy Chọn) Tự Động Ngắt Kết Nối Máy Ảo Để Tiết Kiệm Compute Units
Bỏ comment dòng dưới đây nếu bạn muốn Colab tự động tắt sau khi hoàn thành toàn bộ tác vụ.


In [ ]:
# from google.colab import runtime
# runtime.unassign()
